In [14]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.linear_model import SGDRegressor
from sklearn.metrics import mean_squared_error

# load dataset
path = "./project/Dataset/CSM_dataset.xlsx"
dataset = pd.read_excel(path)

# remove empty rows
dset = dataset.dropna(how='all')
dset.columns = range(dset.shape[1])
dset = dset.drop(columns=[0])

# separate target and features
Y = dset.pop(4)
X = dset

# drop rows where target is NaN
combined = pd.concat([X, Y], axis=1).dropna(subset=[4])
Y = combined.pop(4)
X = combined

# remove outliers using IQR
Q1 = X.quantile(0.25)
Q3 = X.quantile(0.75)
IQR = Q3 - Q1
mask = ~((X < (Q1 - 1.5 * IQR)) | (X > (Q3 + 1.5 * IQR))).any(axis=1)
X = X[mask]
Y = Y[mask]

# convert to numpy
X = X.values
Y = Y.values.reshape(-1, 1)
print('X  :\n',X)
print('Y  :\n',Y)

X  :
 [[2.014e+03 6.300e+00 8.000e+00 ... 4.250e+02 6.360e+02 1.120e+06]
 [2.014e+03 6.200e+00 1.000e+00 ... 3.400e+01 4.700e+01 4.830e+05]
 [2.014e+03 4.600e+00 3.000e+00 ... 7.000e+00 1.000e+00 3.100e+05]
 ...
 [2.015e+03 5.400e+00 8.000e+00 ... 3.250e+02 4.090e+02       nan]
 [2.015e+03 5.400e+00 1.000e+00 ... 6.700e+01 2.010e+02       nan]
 [2.015e+03 4.400e+00 1.500e+01 ... 4.310e+02 6.060e+02       nan]]
Y  :
 [[     9130]
 [ 30700000]
 [    29000]
 [ 42600000]
 [  5750000]
 [ 26000000]
 [350000000]
 [ 15200000]
 [ 85900000]
 [    11800]
 [    72300]
 [ 14600000]
 [ 21600000]
 [ 25400000]
 [ 20300000]
 [  1870000]
 [     9840]
 [  6370000]
 [ 30500000]
 [ 15800000]
 [151000000]
 [ 28800000]
 [ 38900000]
 [ 23400000]
 [     8690]
 [ 85700000]
 [   102000]
 [ 60800000]
 [   104000]
 [    30100]
 [   275000]
 [  8090000]
 [ 47600000]
 [128000000]
 [ 47000000]
 [ 43000000]
 [  2450000]
 [   129000]
 [ 82400000]
 [     8300]
 [     2470]
 [127000000]
 [   348000]
 [ 10400000]
 [112000

In [15]:

# split data
x_train, x_test, y_train, y_test = train_test_split(
    X, Y, test_size=0.2, random_state=10
)

# handle missing values
imputer = SimpleImputer(strategy='mean')
x_train = imputer.fit_transform(x_train)
x_test  = imputer.transform(x_test)
print('\nx_train :\n',x_train)
print('\nx_test :\n',x_test)
# scale X to 0-1
x_scaler = MinMaxScaler()
x_train = x_scaler.fit_transform(x_train)
x_test  = x_scaler.transform(x_test)

# scale y to 0-1
y_scaler = MinMaxScaler()
y_train = y_scaler.fit_transform(y_train).ravel()
y_test  = y_scaler.transform(y_test).ravel()


x_train :
 [[2.01400000e+03 5.70000000e+00 1.00000000e+00 ... 2.55000000e+02
  1.23500000e+03 3.20900000e+06]
 [2.01500000e+03 6.60000000e+00 8.00000000e+00 ... 4.80000000e+02
  1.71200000e+03 1.43958122e+06]
 [2.01400000e+03 7.20000000e+00 1.00000000e+00 ... 4.05000000e+02
  2.73200000e+03 1.43958122e+06]
 ...
 [2.01400000e+03 6.50000000e+00 3.00000000e+00 ... 4.54000000e+02
  1.15000000e+03 4.76910000e+06]
 [2.01500000e+03 6.90000000e+00 8.00000000e+00 ... 2.93000000e+02
  7.00000000e+02 1.43958122e+06]
 [2.01400000e+03 6.60000000e+00 8.00000000e+00 ... 2.47000000e+02
  4.60000000e+02 2.53000000e+05]]

x_test :
 [[ 2.01500000e+03  5.80000000e+00  8.00000000e+00  4.00000000e+07
   2.81600000e+03  1.00000000e+00  3.00000000e+00  3.09874900e+06
   4.31100000e+03  3.41000000e+02  8.81000000e+02  1.52000000e+06]
 [ 2.01400000e+03  6.90000000e+00  3.00000000e+00  2.48782027e+06
   2.00000000e+00  1.00000000e+00  1.10000000e+01  1.66612000e+05
   5.71000000e+02  3.60000000e+01  7.00000000e

In [19]:
# train SGD regressor
sgd = SGDRegressor(max_iter=1000, learning_rate='adaptive', eta0=0.01, random_state=120)
sgd.fit(x_train, y_train)

# predict
y_pred = sgd.predict(x_test)

# calculate MSE
mse = mean_squared_error(y_test, y_pred)
print(f"MSE : {mse}")

MSE : 0.005195857915610846
